In [4]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. LOSS FUNCTION & CHAIN RULE GRADIENTS
# ==========================================
alpha_loss = 24  # Rescaling factor scaled to approx n^(k/2)
beta_loss = 0.5
# Edwards-Erdös bound for the regularization parameter v
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def evaluate_expectations(theta):
    """Helper to get pure expectation values for a single parameter array."""
    job = estimator.run([
        (ansatz, observables_x, theta),
        (ansatz, observables_y, theta),
        (ansatz, observables_z, theta)
    ])
    results = job.result()
    
    # Extract 1D arrays of expectation values
    E_x = results[0].data.evs
    E_y = results[1].data.evs
    E_z = results[2].data.evs
    return np.concatenate([E_x, E_y, E_z])

def calc_loss_and_chain_rule(E):
    """Analytically calculates the loss and the gradient dL/dE."""
    # Precompute common terms for speed
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    # Chain rule for the Cut Loss term
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        # dL/dE for connected nodes
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    # Chain rule for the Regularization term
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    # Add regularization gradient to dL/dE
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
def param_shift_gradient(theta):
    """Calculates the full gradient using batched Parameter Shift + Chain Rule."""
    num_params = len(theta)
    shift = np.pi / 2.0
    
    # Step A: Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) # Shape: (2*num_params, num_params)
    
    # FIX: Add a new axis to force a Cartesian product broadcast
    # Now it evaluates all 240 parameter sets against all 60 observables
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Step B: Run batched estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # Safely extract and orient the results to shape (len(observables), 2*num_params)
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        # Transpose if the shape is (240, 60) to make it (60, 240)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    # Stack vertically to get shape (num_nodes, 2*num_params)
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    # Step C: Compute Jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    # Step D: Apply Chain Rule
    E_current = evaluate_expectations(theta)
    loss, dL_dE = calc_loss_and_chain_rule(E_current)
    grad = dL_dE @ J
    
    # Save for bit-swap decoding
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue.npy"
num_new_params = ansatz.num_parameters # For reps=4 and 12 qubits, this will be 120

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize_param_shift(theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    print(f"Starting optimization with Parameter Shift Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    for i in range(max_iter):
        t += 1
        
        loss, grad = param_shift_gradient(theta_i)
        print(f"--- Adam Step {i+1}/{max_iter} --- | Loss: {loss:.6f}")
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

best_params = adam_optimize_param_shift(
    theta0=initial_params,
    lr=0.1,        
    max_iter=50
)

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue2004.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")

# Single-bit swap search as described in the paper
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue.npy...
Starting optimization with Parameter Shift Adam (lr=0.1)...
--- Adam Step 1/50 --- | Loss: -4773.284460
--- Adam Step 2/50 --- | Loss: -3511.035545
--- Adam Step 3/50 --- | Loss: -4118.387840
--- Adam Step 4/50 --- | Loss: -4506.264308
--- Adam Step 5/50 --- | Loss: -4243.427768
--- Adam Step 6/50 --- | Loss: -4196.784677
--- Adam Step 7/50 --- | Loss: -4383.789716
--- Adam Step 8/50 --- | Loss: -4422.059348
--- Adam Step 9/50 --- | Loss: -4397.440552
--- Adam Step 10/50 --- | Loss: -4509.429775
--- Adam Step 11/50 --- | Loss: -4565.132913
--- Adam Step 12/50 --- | Loss: -4573.084910
--- Adam Step 13/50 --- | Loss: -4548.074734
--- Adam Step 14/50 --- | Loss: -4599.752094
--- Adam Step 15/50 --- | Loss: -4639.931583
--- Adam Step 16/50 --- | Loss: -4646.254254
--- Adam Step 17/50 --- | Loss: -4650.270034
--- Adam Step 18/50 --- | Loss: -4663.628646
--- Adam Step 19/50 --- | Loss: -467

In [5]:
# ==========================================
# 7. DEEP DECODING & BIT-SWAP POST-PROCESSING
# ==========================================

# 1. Extract initial bitstring from the Quantum Circuit's Expectation Values
cur_bits = np.zeros(num_nodes, dtype=int)
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits[i] = 1
    else:
        cur_bits[i] = 0

# Helper to calculate the full absolute cut size once
def calculate_full_cut(graph, bits):
    cut = 0
    for u, v, data in graph.edges(data=True):
        if bits[u] != bits[v]:
            cut += data.get('weight', 1.0)
    return cut

current_cut = calculate_full_cut(graph, cur_bits)
print(f"\nInitial Quantum Cut (Before Local Search): {current_cut}")

# 2. Iterative 1-Bit and 2-Bit Swap Search
search_iteration = 0
improvement_found = True

while improvement_found:
    improvement_found = False
    search_iteration += 1
    
    # --- PHASE 1: 1-BIT SWAP SEARCH ---
    # We sweep through all nodes. If flipping a node improves the cut, we keep it.
    for u in graph.nodes():
        delta = 0
        # Calculate exactly how the cut changes if we flip node `u`
        for v in graph.neighbors(u):
            w = graph[u][v].get('weight', 1.0)
            if cur_bits[u] == cur_bits[v]:
                delta += w  # They were in the same partition. Flipping u cuts this edge.
            else:
                delta -= w  # They were in different partitions. Flipping u un-cuts it.
                
        # If flipping strictly improves the cut, apply the flip
        if delta > 0:
            cur_bits[u] = 1 - cur_bits[u]
            current_cut += delta
            improvement_found = True

    # --- PHASE 2: 2-BIT SWAP SEARCH ---
    # If 1-bit swaps get stuck in a local minimum, we try flipping pairs of connected nodes
    if not improvement_found:
        for u, v in graph.edges():
            delta = 0
            
            # Change in cut for neighbors of u (excluding v)
            for n_u in graph.neighbors(u):
                if n_u == v: continue
                w = graph[u][n_u].get('weight', 1.0)
                if cur_bits[u] == cur_bits[n_u]: delta += w
                else: delta -= w
                    
            # Change in cut for neighbors of v (excluding u)
            for n_v in graph.neighbors(v):
                if n_v == u: continue
                w = graph[v][n_v].get('weight', 1.0)
                if cur_bits[v] == cur_bits[n_v]: delta += w
                else: delta -= w
                    
            if delta > 0:
                cur_bits[u] = 1 - cur_bits[u]
                cur_bits[v] = 1 - cur_bits[v]
                current_cut += delta
                improvement_found = True
                break # Break to restart the 1-bit sweep with the new landscape

print(f"Post-processing complete after {search_iteration} sweeps.")
print(f"Final Deep-Optimized Cut Size: {current_cut}")


Initial Quantum Cut (Before Local Search): 6260.286774176787
Post-processing complete after 12 sweeps.
Final Deep-Optimized Cut Size: 6767.718383338153


In [13]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=3)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. LOSS FUNCTION & CHAIN RULE GRADIENTS
# ==========================================
alpha_loss = 24  # Rescaling factor scaled to approx n^(k/2)
beta_loss = 0.5
# Edwards-Erdös bound for the regularization parameter v
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def evaluate_expectations(theta):
    """Helper to get pure expectation values for a single parameter array."""
    job = estimator.run([
        (ansatz, observables_x, theta),
        (ansatz, observables_y, theta),
        (ansatz, observables_z, theta)
    ])
    results = job.result()
    
    # Extract 1D arrays of expectation values
    E_x = results[0].data.evs
    E_y = results[1].data.evs
    E_z = results[2].data.evs
    return np.concatenate([E_x, E_y, E_z])

def calc_loss_and_chain_rule(E):
    """Analytically calculates the loss and the gradient dL/dE."""
    # Precompute common terms for speed
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    # Chain rule for the Cut Loss term
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        # dL/dE for connected nodes
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    # Chain rule for the Regularization term
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    # Add regularization gradient to dL/dE
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
def param_shift_gradient(theta):
    """Calculates the full gradient using batched Parameter Shift + Chain Rule."""
    num_params = len(theta)
    shift = np.pi / 2.0
    
    # Step A: Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) # Shape: (2*num_params, num_params)
    
    # FIX: Add a new axis to force a Cartesian product broadcast
    # Now it evaluates all 240 parameter sets against all 60 observables
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Step B: Run batched estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # Safely extract and orient the results to shape (len(observables), 2*num_params)
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        # Transpose if the shape is (240, 60) to make it (60, 240)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    # Stack vertically to get shape (num_nodes, 2*num_params)
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    # Step C: Compute Jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    # Step D: Apply Chain Rule
    E_current = evaluate_expectations(theta)
    loss, dL_dE = calc_loss_and_chain_rule(E_current)
    grad = dL_dE @ J
    
    # Save for bit-swap decoding
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue2004.npy"
num_new_params = ansatz.num_parameters # For reps=4 and 12 qubits, this will be 120

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize_param_shift(theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    print(f"Starting optimization with Parameter Shift Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    for i in range(max_iter):
        t += 1
        
        loss, grad = param_shift_gradient(theta_i)
        print(f"--- Adam Step {i+1}/{max_iter} --- | Loss: {loss:.6f}")
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

best_params = adam_optimize_param_shift(
    theta0=initial_params,
    lr=0.1,        
    max_iter=100
)

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue2008.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")

# Single-bit swap search as described in the paper
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue2004.npy...
Starting optimization with Parameter Shift Adam (lr=0.1)...
--- Adam Step 1/100 --- | Loss: 616.343729
--- Adam Step 2/100 --- | Loss: -114.224595
--- Adam Step 3/100 --- | Loss: -364.233930
--- Adam Step 4/100 --- | Loss: -637.410922
--- Adam Step 5/100 --- | Loss: -836.684283
--- Adam Step 6/100 --- | Loss: -1005.404363
--- Adam Step 7/100 --- | Loss: -1083.221177
--- Adam Step 8/100 --- | Loss: -1264.708236
--- Adam Step 9/100 --- | Loss: -1526.582524
--- Adam Step 10/100 --- | Loss: -1758.157458
--- Adam Step 11/100 --- | Loss: -1929.739860
--- Adam Step 12/100 --- | Loss: -2082.603172
--- Adam Step 13/100 --- | Loss: -2232.904240
--- Adam Step 14/100 --- | Loss: -2237.285385
--- Adam Step 15/100 --- | Loss: -2453.446880
--- Adam Step 16/100 --- | Loss: -2489.449176
--- Adam Step 17/100 --- | Loss: -2610.750308
--- Adam Step 18/100 --- | Loss: -2713.116819
--- Adam Step 19/100

In [15]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=2)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. LOSS FUNCTION & CHAIN RULE GRADIENTS
# ==========================================
alpha_loss = 24  # Rescaling factor scaled to approx n^(k/2)
beta_loss = 0.5
# Edwards-Erdös bound for the regularization parameter v
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def evaluate_expectations(theta):
    """Helper to get pure expectation values for a single parameter array."""
    job = estimator.run([
        (ansatz, observables_x, theta),
        (ansatz, observables_y, theta),
        (ansatz, observables_z, theta)
    ])
    results = job.result()
    
    # Extract 1D arrays of expectation values
    E_x = results[0].data.evs
    E_y = results[1].data.evs
    E_z = results[2].data.evs
    return np.concatenate([E_x, E_y, E_z])

def calc_loss_and_chain_rule(E):
    """Analytically calculates the loss and the gradient dL/dE."""
    # Precompute common terms for speed
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    # Chain rule for the Cut Loss term
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        # dL/dE for connected nodes
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    # Chain rule for the Regularization term
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    # Add regularization gradient to dL/dE
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
def param_shift_gradient(theta):
    """Calculates the full gradient using batched Parameter Shift + Chain Rule."""
    num_params = len(theta)
    shift = np.pi / 2.0
    
    # Step A: Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) # Shape: (2*num_params, num_params)
    
    # FIX: Add a new axis to force a Cartesian product broadcast
    # Now it evaluates all 240 parameter sets against all 60 observables
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Step B: Run batched estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # Safely extract and orient the results to shape (len(observables), 2*num_params)
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        # Transpose if the shape is (240, 60) to make it (60, 240)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    # Stack vertically to get shape (num_nodes, 2*num_params)
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    # Step C: Compute Jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    # Step D: Apply Chain Rule
    E_current = evaluate_expectations(theta)
    loss, dL_dE = calc_loss_and_chain_rule(E_current)
    grad = dL_dE @ J
    
    # Save for bit-swap decoding
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue2004.npy"
num_new_params = ansatz.num_parameters # For reps=4 and 12 qubits, this will be 120

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize_param_shift(theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    print(f"Starting optimization with Parameter Shift Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    for i in range(max_iter):
        t += 1
        
        loss, grad = param_shift_gradient(theta_i)
        print(f"--- Adam Step {i+1}/{max_iter} --- | Loss: {loss:.6f}")
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

best_params = adam_optimize_param_shift(
    theta0=initial_params,
    lr=0.1,        
    max_iter=100
)

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue2010.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")

# Single-bit swap search as described in the paper
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue2004.npy...
Starting optimization with Parameter Shift Adam (lr=0.1)...
--- Adam Step 1/100 --- | Loss: 2828.207425
--- Adam Step 2/100 --- | Loss: 1065.064261
--- Adam Step 3/100 --- | Loss: -230.232288
--- Adam Step 4/100 --- | Loss: -636.177991
--- Adam Step 5/100 --- | Loss: -690.134686
--- Adam Step 6/100 --- | Loss: -800.530377
--- Adam Step 7/100 --- | Loss: -979.749655
--- Adam Step 8/100 --- | Loss: -1227.191015
--- Adam Step 9/100 --- | Loss: -1378.863477
--- Adam Step 10/100 --- | Loss: -1496.078216
--- Adam Step 11/100 --- | Loss: -1647.933020
--- Adam Step 12/100 --- | Loss: -1834.625647
--- Adam Step 13/100 --- | Loss: -1952.003822
--- Adam Step 14/100 --- | Loss: -2005.149053
--- Adam Step 15/100 --- | Loss: -2033.778918
--- Adam Step 16/100 --- | Loss: -2080.502645
--- Adam Step 17/100 --- | Loss: -2161.809611
--- Adam Step 18/100 --- | Loss: -2255.774125
--- Adam Step 19/100 

In [16]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=2)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. LOSS FUNCTION & CHAIN RULE GRADIENTS
# ==========================================
alpha_loss = 24  # Rescaling factor scaled to approx n^(k/2)
beta_loss = 0.5
# Edwards-Erdös bound for the regularization parameter v
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def evaluate_expectations(theta):
    """Helper to get pure expectation values for a single parameter array."""
    job = estimator.run([
        (ansatz, observables_x, theta),
        (ansatz, observables_y, theta),
        (ansatz, observables_z, theta)
    ])
    results = job.result()
    
    # Extract 1D arrays of expectation values
    E_x = results[0].data.evs
    E_y = results[1].data.evs
    E_z = results[2].data.evs
    return np.concatenate([E_x, E_y, E_z])

def calc_loss_and_chain_rule(E):
    """Analytically calculates the loss and the gradient dL/dE."""
    # Precompute common terms for speed
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    # Chain rule for the Cut Loss term
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        # dL/dE for connected nodes
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    # Chain rule for the Regularization term
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    # Add regularization gradient to dL/dE
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
def param_shift_gradient(theta):
    """Calculates the full gradient using batched Parameter Shift + Chain Rule."""
    num_params = len(theta)
    shift = np.pi / 2.0
    
    # Step A: Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) # Shape: (2*num_params, num_params)
    
    # FIX: Add a new axis to force a Cartesian product broadcast
    # Now it evaluates all 240 parameter sets against all 60 observables
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Step B: Run batched estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # Safely extract and orient the results to shape (len(observables), 2*num_params)
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        # Transpose if the shape is (240, 60) to make it (60, 240)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    # Stack vertically to get shape (num_nodes, 2*num_params)
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    # Step C: Compute Jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    # Step D: Apply Chain Rule
    E_current = evaluate_expectations(theta)
    loss, dL_dE = calc_loss_and_chain_rule(E_current)
    grad = dL_dE @ J
    
    # Save for bit-swap decoding
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue2010.npy"
num_new_params = ansatz.num_parameters # For reps=4 and 12 qubits, this will be 120

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize_param_shift(theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    print(f"Starting optimization with Parameter Shift Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    for i in range(max_iter):
        t += 1
        
        loss, grad = param_shift_gradient(theta_i)
        print(f"--- Adam Step {i+1}/{max_iter} --- | Loss: {loss:.6f}")
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

best_params = adam_optimize_param_shift(
    theta0=initial_params,
    lr=0.1,        
    max_iter=50
)

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue2010_2.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")

# Single-bit swap search as described in the paper
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue2010.npy...
Starting optimization with Parameter Shift Adam (lr=0.1)...
--- Adam Step 1/50 --- | Loss: -3980.688920
--- Adam Step 2/50 --- | Loss: -2880.510648
--- Adam Step 3/50 --- | Loss: -3597.048787
--- Adam Step 4/50 --- | Loss: -3556.225365
--- Adam Step 5/50 --- | Loss: -3541.422493
--- Adam Step 6/50 --- | Loss: -3704.888923
--- Adam Step 7/50 --- | Loss: -3798.975237
--- Adam Step 8/50 --- | Loss: -3797.528720
--- Adam Step 9/50 --- | Loss: -3790.968308
--- Adam Step 10/50 --- | Loss: -3826.161276
--- Adam Step 11/50 --- | Loss: -3892.299657
--- Adam Step 12/50 --- | Loss: -3935.675475
--- Adam Step 13/50 --- | Loss: -3930.374342
--- Adam Step 14/50 --- | Loss: -3952.520408
--- Adam Step 15/50 --- | Loss: -3989.220142
--- Adam Step 16/50 --- | Loss: -4011.468990
--- Adam Step 17/50 --- | Loss: -4007.546240
--- Adam Step 18/50 --- | Loss: -3996.875768
--- Adam Step 19/50 --- | Loss: 

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=2)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. LOSS FUNCTION & CHAIN RULE GRADIENTS
# ==========================================
alpha_loss = 24  # Rescaling factor scaled to approx n^(k/2)
beta_loss = 0.5
# Edwards-Erdös bound for the regularization parameter v
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def evaluate_expectations(theta):
    """Helper to get pure expectation values for a single parameter array."""
    job = estimator.run([
        (ansatz, observables_x, theta),
        (ansatz, observables_y, theta),
        (ansatz, observables_z, theta)
    ])
    results = job.result()
    
    # Extract 1D arrays of expectation values
    E_x = results[0].data.evs
    E_y = results[1].data.evs
    E_z = results[2].data.evs
    return np.concatenate([E_x, E_y, E_z])

def calc_loss_and_chain_rule(E):
    """Analytically calculates the loss and the gradient dL/dE."""
    # Precompute common terms for speed
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    # Chain rule for the Cut Loss term
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        # dL/dE for connected nodes
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    # Chain rule for the Regularization term
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    # Add regularization gradient to dL/dE
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
def param_shift_gradient(theta):
    """Calculates the full gradient using batched Parameter Shift + Chain Rule."""
    num_params = len(theta)
    shift = np.pi / 2.0
    
    # Step A: Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) # Shape: (2*num_params, num_params)
    
    # FIX: Add a new axis to force a Cartesian product broadcast
    # Now it evaluates all 240 parameter sets against all 60 observables
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Step B: Run batched estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # Safely extract and orient the results to shape (len(observables), 2*num_params)
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        # Transpose if the shape is (240, 60) to make it (60, 240)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    # Stack vertically to get shape (num_nodes, 2*num_params)
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    # Step C: Compute Jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    # Step D: Apply Chain Rule
    E_current = evaluate_expectations(theta)
    loss, dL_dE = calc_loss_and_chain_rule(E_current)
    grad = dL_dE @ J
    
    # Save for bit-swap decoding
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue2010.npy"
num_new_params = ansatz.num_parameters # For reps=4 and 12 qubits, this will be 120

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize_param_shift(theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    print(f"Starting optimization with Parameter Shift Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    for i in range(max_iter):
        t += 1
        
        loss, grad = param_shift_gradient(theta_i)
        print(f"--- Adam Step {i+1}/{max_iter} --- | Loss: {loss:.6f}")
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

best_params = adam_optimize_param_shift(
    theta0=initial_params,
    lr=0.1,        
    max_iter=1000
)

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue2010_linear.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")

# Single-bit swap search as described in the paper
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...


/tmp/ipykernel_200518/2770506738.py:59: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=2)


Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue2010.npy...
Starting optimization with Parameter Shift Adam (lr=0.1)...
--- Adam Step 1/1000 --- | Loss: -414.827729
--- Adam Step 2/1000 --- | Loss: -1443.805246
--- Adam Step 3/1000 --- | Loss: -2379.052331
--- Adam Step 4/1000 --- | Loss: -2717.246694
--- Adam Step 5/1000 --- | Loss: -3045.939388
--- Adam Step 6/1000 --- | Loss: -3200.582885
--- Adam Step 7/1000 --- | Loss: -3258.536080
--- Adam Step 8/1000 --- | Loss: -3320.016664
--- Adam Step 9/1000 --- | Loss: -3405.043349
--- Adam Step 10/1000 --- | Loss: -3444.465676
--- Adam Step 11/1000 --- | Loss: -3481.498053
--- Adam Step 12/1000 --- | Loss: -3531.447777
--- Adam Step 13/1000 --- | Loss: -3576.887886
--- Adam Step 14/1000 --- | Loss: -3612.848341
--- Adam Step 15/1000 --- | Loss: -3628.925479
--- Adam Step 16/1000 --- | Loss: -3683.923472
--- Adam Step 17/1000 --- | Loss: -3717.085261
--- Adam Step 18/1000 --- | Loss: -3745.519795
--- Adam Step 19

In [2]:
# ==========================================
# 7. DEEP DECODING & BIT-SWAP POST-PROCESSING
# ==========================================

# 1. Extract initial bitstring from the Quantum Circuit's Expectation Values
cur_bits = np.zeros(num_nodes, dtype=int)
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits[i] = 1
    else:
        cur_bits[i] = 0

# Helper to calculate the full absolute cut size once
def calculate_full_cut(graph, bits):
    cut = 0
    for u, v, data in graph.edges(data=True):
        if bits[u] != bits[v]:
            cut += data.get('weight', 1.0)
    return cut

current_cut = calculate_full_cut(graph, cur_bits)
print(f"\nInitial Quantum Cut (Before Local Search): {current_cut}")

# 2. Iterative 1-Bit and 2-Bit Swap Search
search_iteration = 0
improvement_found = True

while improvement_found:
    improvement_found = False
    search_iteration += 1
    
    # --- PHASE 1: 1-BIT SWAP SEARCH ---
    # We sweep through all nodes. If flipping a node improves the cut, we keep it.
    for u in graph.nodes():
        delta = 0
        # Calculate exactly how the cut changes if we flip node `u`
        for v in graph.neighbors(u):
            w = graph[u][v].get('weight', 1.0)
            if cur_bits[u] == cur_bits[v]:
                delta += w  # They were in the same partition. Flipping u cuts this edge.
            else:
                delta -= w  # They were in different partitions. Flipping u un-cuts it.
                
        # If flipping strictly improves the cut, apply the flip
        if delta > 0:
            cur_bits[u] = 1 - cur_bits[u]
            current_cut += delta
            improvement_found = True

    # --- PHASE 2: 2-BIT SWAP SEARCH ---
    # If 1-bit swaps get stuck in a local minimum, we try flipping pairs of connected nodes
    if not improvement_found:
        for u, v in graph.edges():
            delta = 0
            
            # Change in cut for neighbors of u (excluding v)
            for n_u in graph.neighbors(u):
                if n_u == v: continue
                w = graph[u][n_u].get('weight', 1.0)
                if cur_bits[u] == cur_bits[n_u]: delta += w
                else: delta -= w
                    
            # Change in cut for neighbors of v (excluding u)
            for n_v in graph.neighbors(v):
                if n_v == u: continue
                w = graph[v][n_v].get('weight', 1.0)
                if cur_bits[v] == cur_bits[n_v]: delta += w
                else: delta -= w
                    
            if delta > 0:
                cur_bits[u] = 1 - cur_bits[u]
                cur_bits[v] = 1 - cur_bits[v]
                current_cut += delta
                improvement_found = True
                break # Break to restart the 1-bit sweep with the new landscape

print(f"Post-processing complete after {search_iteration} sweeps.")
print(f"Final Deep-Optimized Cut Size: {current_cut}")


Initial Quantum Cut (Before Local Search): 6220.84656562341
Post-processing complete after 11 sweeps.
Final Deep-Optimized Cut Size: 6721.241370628416


In [12]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. LOSS FUNCTION & CHAIN RULE GRADIENTS
# ==========================================
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

# ... (evaluate_expectations remains unchanged) ...

def calc_loss_and_chain_rule(E, alpha_loss): # <--- Added alpha_loss
    """Analytically calculates the loss and the gradient dL/dE."""
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

def compute_loss_only(theta, alpha_loss): 
    """Evaluates the loss for a given theta without computing gradients."""
    E = evaluate_expectations(theta)
    loss, _ = calc_loss_and_chain_rule(E, alpha_loss)
    return loss



# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
def param_shift_gradient(theta, alpha_loss): # <--- Added alpha_loss parameter
    """Calculates the full gradient using batched Parameter Shift + Chain Rule."""
    num_params = len(theta)
    shift = np.pi / 2.0
    
    # Step A: Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) # Shape: (2*num_params, num_params)
    
    # Add a new axis to force a Cartesian product broadcast
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Step B: Run batched estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # Safely extract and orient the results to shape (len(observables), 2*num_params)
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    # Stack vertically to get shape (num_nodes, 2*num_params)
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    # Step C: Compute Jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    # Step D: Apply Chain Rule
    E_current = evaluate_expectations(theta)
    
    # Pass the dynamic alpha_loss into the chain rule calculation
    loss, dL_dE = calc_loss_and_chain_rule(E_current, alpha_loss) # <--- Passed here
    grad = dL_dE @ J
    
    # Save for bit-swap decoding
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad
# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue2010_linear4.npy"
num_new_params = ansatz.num_parameters # For reps=4 and 12 qubits, this will be 120

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

def adam_optimize_param_shift_armijo(theta0, initial_lr=0.5, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50, alpha_start=1.5, alpha_end=12.0):
    print(f"Starting optimization with Parameter Shift Adam + Armijo Line Search + Alpha Annealing...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    c1 = 1e-4
    tau = 0.5
    max_ls_iter = 10 
    
    for i in range(max_iter):
        t += 1
        
        # --- DYNAMIC ALPHA CALCULATION (Linear Annealing) ---
        # Starts at alpha_start, reaches alpha_end by the last iteration
        progress = i / max(1, max_iter - 1)
        current_alpha = alpha_start + (alpha_end - alpha_start) * progress
        
        # 1. Get current loss and full gradient using dynamic alpha
        current_loss, grad = param_shift_gradient(theta_i, current_alpha)
        
        # 2. Compute Adam moments
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        # 3. Define the descent direction
        step_direction = - m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        # 4. Armijo Backtracking Line Search
        alpha_lr = initial_lr
        expected_decrease_rate = c1 * np.dot(grad, step_direction)
        
        ls_iter = 0
        while ls_iter < max_ls_iter:
            theta_candidate = theta_i + alpha_lr * step_direction
            loss_candidate = compute_loss_only(theta_candidate, current_alpha) # Pass current_alpha here
            
            if loss_candidate <= current_loss + alpha_lr * expected_decrease_rate:
                break 
            
            alpha_lr *= tau 
            ls_iter += 1
            
        print(f"--- Adam Step {i+1}/{max_iter} | Alpha: {current_alpha:.2f} | Loss: {current_loss:.6f} | LR: {alpha_lr:.6f} (LS iters: {ls_iter})")
        
        # 5. Apply the step
        theta_next = theta_i + alpha_lr * step_direction
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

# Trigger the run:
best_params = adam_optimize_param_shift_armijo(
    theta0=initial_params,
    initial_lr=0.5,
    max_iter=100,       # Make sure max_iter is high enough for the annealing to be gradual
    alpha_start=1.5,   # Low enough to prevent vanishing gradients early on
    alpha_end=24.0     # High enough to force strict partitions at the end
)

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue2010_linear6_zrzmijo_dynamic_alpha.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")

# Single-bit swap search as described in the paper
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue2010_linear4.npy...
Starting optimization with Parameter Shift Adam + Armijo Line Search + Alpha Annealing...


/tmp/ipykernel_200518/1364550082.py:59: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)


--- Adam Step 1/100 | Alpha: 1.50 | Loss: -144.897946 | LR: 0.125000 (LS iters: 2)
--- Adam Step 2/100 | Alpha: 1.73 | Loss: -232.650384 | LR: 0.250000 (LS iters: 1)
--- Adam Step 3/100 | Alpha: 1.95 | Loss: -305.216060 | LR: 0.250000 (LS iters: 1)
--- Adam Step 4/100 | Alpha: 2.18 | Loss: -480.299076 | LR: 0.250000 (LS iters: 1)
--- Adam Step 5/100 | Alpha: 2.41 | Loss: -641.935221 | LR: 0.250000 (LS iters: 1)
--- Adam Step 6/100 | Alpha: 2.64 | Loss: -813.494906 | LR: 0.250000 (LS iters: 1)
--- Adam Step 7/100 | Alpha: 2.86 | Loss: -1013.884633 | LR: 0.250000 (LS iters: 1)
--- Adam Step 8/100 | Alpha: 3.09 | Loss: -1195.064811 | LR: 0.250000 (LS iters: 1)
--- Adam Step 9/100 | Alpha: 3.32 | Loss: -1466.934117 | LR: 0.250000 (LS iters: 1)
--- Adam Step 10/100 | Alpha: 3.55 | Loss: -1633.001747 | LR: 0.250000 (LS iters: 1)
--- Adam Step 11/100 | Alpha: 3.77 | Loss: -1817.150514 | LR: 0.250000 (LS iters: 1)
--- Adam Step 12/100 | Alpha: 4.00 | Loss: -1988.836457 | LR: 0.250000 (LS iters

In [13]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. LOSS FUNCTION & CHAIN RULE GRADIENTS
# ==========================================
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

# ... (evaluate_expectations remains unchanged) ...

def calc_loss_and_chain_rule(E, alpha_loss): # <--- Added alpha_loss
    """Analytically calculates the loss and the gradient dL/dE."""
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

def compute_loss_only(theta, alpha_loss): 
    """Evaluates the loss for a given theta without computing gradients."""
    E = evaluate_expectations(theta)
    loss, _ = calc_loss_and_chain_rule(E, alpha_loss)
    return loss



# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
def param_shift_gradient(theta, alpha_loss): # <--- Added alpha_loss parameter
    """Calculates the full gradient using batched Parameter Shift + Chain Rule."""
    num_params = len(theta)
    shift = np.pi / 2.0
    
    # Step A: Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) # Shape: (2*num_params, num_params)
    
    # Add a new axis to force a Cartesian product broadcast
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Step B: Run batched estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # Safely extract and orient the results to shape (len(observables), 2*num_params)
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    # Stack vertically to get shape (num_nodes, 2*num_params)
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    # Step C: Compute Jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    # Step D: Apply Chain Rule
    E_current = evaluate_expectations(theta)
    
    # Pass the dynamic alpha_loss into the chain rule calculation
    loss, dL_dE = calc_loss_and_chain_rule(E_current, alpha_loss) # <--- Passed here
    grad = dL_dE @ J
    
    # Save for bit-swap decoding
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad
# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue2010_linear3qbts.npy"
num_new_params = ansatz.num_parameters # For reps=4 and 12 qubits, this will be 120

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

def adam_optimize_param_shift_armijo(theta0, initial_lr=0.5, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50, alpha_start=1.5, alpha_end=12.0):
    print(f"Starting optimization with Parameter Shift Adam + Armijo Line Search + Alpha Annealing...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    c1 = 1e-4
    tau = 0.5
    max_ls_iter = 10 
    
    for i in range(max_iter):
        t += 1
        
        # --- DYNAMIC ALPHA CALCULATION (Linear Annealing) ---
        # Starts at alpha_start, reaches alpha_end by the last iteration
        progress = i / max(1, max_iter - 1)
        current_alpha = alpha_start + (alpha_end - alpha_start) * progress
        
        # 1. Get current loss and full gradient using dynamic alpha
        current_loss, grad = param_shift_gradient(theta_i, current_alpha)
        
        # 2. Compute Adam moments
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        # 3. Define the descent direction
        step_direction = - m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        # 4. Armijo Backtracking Line Search
        alpha_lr = initial_lr
        expected_decrease_rate = c1 * np.dot(grad, step_direction)
        
        ls_iter = 0
        while ls_iter < max_ls_iter:
            theta_candidate = theta_i + alpha_lr * step_direction
            loss_candidate = compute_loss_only(theta_candidate, current_alpha) # Pass current_alpha here
            
            if loss_candidate <= current_loss + alpha_lr * expected_decrease_rate:
                break 
            
            alpha_lr *= tau 
            ls_iter += 1
            
        print(f"--- Adam Step {i+1}/{max_iter} | Alpha: {current_alpha:.2f} | Loss: {current_loss:.6f} | LR: {alpha_lr:.6f} (LS iters: {ls_iter})")
        
        # 5. Apply the step
        theta_next = theta_i + alpha_lr * step_direction
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

# Trigger the run:
best_params = adam_optimize_param_shift_armijo(
    theta0=initial_params,
    initial_lr=0.5,
    max_iter=400,       # Make sure max_iter is high enough for the annealing to be gradual
    alpha_start=1.5,   # Low enough to prevent vanishing gradients early on
    alpha_end=24.0     # High enough to force strict partitions at the end
)

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue2010_linear6_zrzmijo_dynamic_alpha.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")

# Single-bit swap search as described in the paper
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue2010_linear3qbts.npy...
Padding parameters! Transferring 96 angles into a 120 angle circuit.
Starting optimization with Parameter Shift Adam + Armijo Line Search + Alpha Annealing...


/tmp/ipykernel_200518/4249785430.py:59: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)


--- Adam Step 1/400 | Alpha: 1.50 | Loss: -10.550217 | LR: 0.250000 (LS iters: 1)
--- Adam Step 2/400 | Alpha: 1.56 | Loss: -47.568218 | LR: 0.250000 (LS iters: 1)
--- Adam Step 3/400 | Alpha: 1.61 | Loss: -94.846858 | LR: 0.500000 (LS iters: 0)
--- Adam Step 4/400 | Alpha: 1.67 | Loss: -108.525194 | LR: 0.500000 (LS iters: 0)
--- Adam Step 5/400 | Alpha: 1.73 | Loss: -179.408385 | LR: 0.500000 (LS iters: 0)
--- Adam Step 6/400 | Alpha: 1.78 | Loss: -234.752572 | LR: 0.500000 (LS iters: 0)
--- Adam Step 7/400 | Alpha: 1.84 | Loss: -362.603211 | LR: 0.500000 (LS iters: 0)
--- Adam Step 8/400 | Alpha: 1.89 | Loss: -410.155312 | LR: 0.500000 (LS iters: 0)
--- Adam Step 9/400 | Alpha: 1.95 | Loss: -517.730690 | LR: 0.500000 (LS iters: 0)
--- Adam Step 10/400 | Alpha: 2.01 | Loss: -637.144596 | LR: 0.250000 (LS iters: 1)
--- Adam Step 11/400 | Alpha: 2.06 | Loss: -818.279279 | LR: 0.250000 (LS iters: 1)
--- Adam Step 12/400 | Alpha: 2.12 | Loss: -851.297543 | LR: 0.250000 (LS iters: 1)
--- 

In [14]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 9                 # Increased to 12 to support 180 nodes at k=2
k = 3                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=2)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. LOSS FUNCTION & CHAIN RULE GRADIENTS
# ==========================================
alpha_loss = 24  # Rescaling factor scaled to approx n^(k/2)
beta_loss = 0.5
# Edwards-Erdös bound for the regularization parameter v
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def evaluate_expectations(theta):
    """Helper to get pure expectation values for a single parameter array."""
    job = estimator.run([
        (ansatz, observables_x, theta),
        (ansatz, observables_y, theta),
        (ansatz, observables_z, theta)
    ])
    results = job.result()
    
    # Extract 1D arrays of expectation values
    E_x = results[0].data.evs
    E_y = results[1].data.evs
    E_z = results[2].data.evs
    return np.concatenate([E_x, E_y, E_z])

def calc_loss_and_chain_rule(E):
    """Analytically calculates the loss and the gradient dL/dE."""
    # Precompute common terms for speed
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    # Chain rule for the Cut Loss term
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        # dL/dE for connected nodes
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    # Chain rule for the Regularization term
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    # Add regularization gradient to dL/dE
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
def param_shift_gradient(theta):
    """Calculates the full gradient using batched Parameter Shift + Chain Rule."""
    num_params = len(theta)
    shift = np.pi / 2.0
    
    # Step A: Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) # Shape: (2*num_params, num_params)
    
    # FIX: Add a new axis to force a Cartesian product broadcast
    # Now it evaluates all 240 parameter sets against all 60 observables
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Step B: Run batched estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # Safely extract and orient the results to shape (len(observables), 2*num_params)
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        # Transpose if the shape is (240, 60) to make it (60, 240)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    # Stack vertically to get shape (num_nodes, 2*num_params)
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    # Step C: Compute Jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    # Step D: Apply Chain Rule
    E_current = evaluate_expectations(theta)
    loss, dL_dE = calc_loss_and_chain_rule(E_current)
    grad = dL_dE @ J
    
    # Save for bit-swap decoding
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue2010.npy"
num_new_params = ansatz.num_parameters # For reps=4 and 12 qubits, this will be 120

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize_param_shift(theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    print(f"Starting optimization with Parameter Shift Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    for i in range(max_iter):
        t += 1
        
        loss, grad = param_shift_gradient(theta_i)
        print(f"--- Adam Step {i+1}/{max_iter} --- | Loss: {loss:.6f}")
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

best_params = adam_optimize_param_shift(
    theta0=initial_params,
    lr=0.1,        
    max_iter=100
)

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue2010_linear_k3.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")

# Single-bit swap search as described in the paper
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue2010.npy...
Starting optimization with Parameter Shift Adam (lr=0.1)...


/tmp/ipykernel_200518/3190795400.py:59: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=2)


--- Adam Step 1/100 --- | Loss: 1071.932672
--- Adam Step 2/100 --- | Loss: 536.778641
--- Adam Step 3/100 --- | Loss: 200.140639
--- Adam Step 4/100 --- | Loss: -254.279292
--- Adam Step 5/100 --- | Loss: -321.672022
--- Adam Step 6/100 --- | Loss: -530.438524
--- Adam Step 7/100 --- | Loss: -666.605068
--- Adam Step 8/100 --- | Loss: -875.726446
--- Adam Step 9/100 --- | Loss: -1168.803780
--- Adam Step 10/100 --- | Loss: -1466.613613
--- Adam Step 11/100 --- | Loss: -1744.922227
--- Adam Step 12/100 --- | Loss: -1814.616532
--- Adam Step 13/100 --- | Loss: -1911.192395
--- Adam Step 14/100 --- | Loss: -2071.637955
--- Adam Step 15/100 --- | Loss: -2273.143713
--- Adam Step 16/100 --- | Loss: -2426.300429
--- Adam Step 17/100 --- | Loss: -2484.228735
--- Adam Step 18/100 --- | Loss: -2605.280211
--- Adam Step 19/100 --- | Loss: -2730.221597
--- Adam Step 20/100 --- | Loss: -2782.790050
--- Adam Step 21/100 --- | Loss: -2822.780100
--- Adam Step 22/100 --- | Loss: -2918.623864
--- Ada

In [15]:
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pyquil.paulis import sI, sX, sY, sZ
from pytket.passes import AutoRebase
from pyquil.api import WavefunctionSimulator
import numpy as np
from pytket.circuit import OpType
from pyquil.quilbase import Gate
from pyquil import Program, get_qc
from pyquil.gates import H, RX, MEASURE
from pyquil.quil import address_qubits

# Import Mitiq for Pauli Twirling
#from mitiq import rc

# ==========================================
# 8. PAULI TWIRLING & TRANSLATE VIA PYTKET
# ==========================================
# ... (your previous imports) ...
from pyquil.quil import address_qubits

# Corrected Import for Mitiq Pauli Twirling
from mitiq import pt

# ==========================================
# 8. PAULI TWIRLING & TRANSLATE VIA PYTKET
# ==========================================
print("\nApplying Pauli Twirling and translating to PyQuil...")

ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=2)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])
best_params_loaded = np.load("pce_optimal_theta_adam_reps4qbt12Continue2010_linear_k3.npy")
bound_circuit = ansatz.assign_parameters(best_params_loaded)

# --- MITIQ: Generate twirled ensemble ---
num_rc_circuits = 6
print(f"Generating {num_rc_circuits} twirled circuits...")

# Corrected Mitiq function call
twirled_qiskit_circuits = pt.generate_pauli_twirl_variants(
    bound_circuit, 
    num_circuits=num_rc_circuits
)

rebase = AutoRebase({OpType.CX, OpType.Rx, OpType.Rz})
pyquil_rc_progs = []

# Translate each twirled circuit to PyQuil
for circ in twirled_qiskit_circuits:
    tket_circ = qiskit_to_tk(circ)
    rebase.apply(tket_circ)
    pyquil_rc_progs.append(tk_to_pyquil(tket_circ))

def qiskit_to_pyquil_pauli(qiskit_op):
    pauli_str = qiskit_op.paulis.to_labels()[0]
    term = sI() 
    for qubit_idx, char in enumerate(reversed(pauli_str)):
        if char == 'X':
            term *= sX(qubit_idx)
        elif char == 'Y':
            term *= sY(qubit_idx)
        elif char == 'Z':
            term *= sZ(qubit_idx)
    return term

print("Translating observables...")
pyquil_obs_x = [qiskit_to_pyquil_pauli(op) for op in observables_x]
pyquil_obs_y = [qiskit_to_pyquil_pauli(op) for op in observables_y]
pyquil_obs_z = [qiskit_to_pyquil_pauli(op) for op in observables_z]

# ==========================================
# 9. EVALUATE ON PYQUIL QVM & DECODE
# ==========================================
print("Evaluating exact expectation values on PyQuil QVM...")

wfn_sim = WavefunctionSimulator()

# For the QVM, evaluating just the first circuit is sufficient 
# since simulators without noise don't suffer from coherent errors
E_x_qvm = np.real(wfn_sim.expectation(pyquil_rc_progs[0], pauli_terms=pyquil_obs_x))
E_y_qvm = np.real(wfn_sim.expectation(pyquil_rc_progs[0], pauli_terms=pyquil_obs_y))
E_z_qvm = np.real(wfn_sim.expectation(pyquil_rc_progs[0], pauli_terms=pyquil_obs_z))

E_final_qvm = np.concatenate([E_x_qvm, E_y_qvm, E_z_qvm])

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = [1 if val >= 0 else 0 for val in E_final_qvm]
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

# Local Search (QVM)
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post QVM):", best_cut)



Applying Pauli Twirling and translating to PyQuil...
Generating 6 twirled circuits...


/tmp/ipykernel_200518/1755906973.py:30: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=2)


Translating observables...
Evaluating exact expectation values on PyQuil QVM...
Final Optimized Cut Size (Post QVM): 6547.23533894701


In [17]:
# ==========================================
# 10. APPLY MAPPING & EXECUTE ON ANKAA-3
# ==========================================
from pyquil.quil import Program
#from pyquil.quilatom import Pragma
from pyquil.gates import H, RX, MEASURE
import numpy as np

print("Preparing twirled ensemble for Rigetti Ankaa-3...")

# 🏆 The Optimal Comprehensive Chain mapping (Highest T1/T2, f1Q, fRO, fISWAP)
qubit_mapping = {
    0: 2, 
    1: 9, 
    2: 8, 
    3: 7, 
    4: 14, 
    5: 21, 
    6: 22, 
    7: 15, 
    8: 16
}

qc = get_qc("Ankaa-3")
total_shots = 10000
# Distribute shots evenly across the RC ensemble
shots_per_circ = total_shots // num_rc_circuits

def run_mapped_basis(mapped_prog, basis, mapping, qc, shots):
    # FORCE the compiler to use our exact physical qubits, no auto-rerouting!
    prog = Program(f'PRAGMA INITIAL_REWIRE "NAIVE"')
    prog += mapped_prog
    
    # Declare the readout register (ro) sized to our logical circuit
    ro = prog.declare("ro", "BIT", len(mapping))
    
    # Append the basis transformation gates to the physical qubits
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
    # Measure the physical qubits into the logical indices of the readout register
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    exe = qc.compile(prog)
    result = qc.run(exe)
    
    # PyQuil 3.x+ syntax for extracting readout results
    if hasattr(result, "readout_data"):
        return result.readout_data.get("ro") # PyQuil 4.x
    else:
        return result.get_register_map().get("ro") # Older PyQuil 3.x

def get_expectations(bitstrings, qiskit_observables, shots):
    # Convert {0, 1} bitstrings to {+1, -1} spin expectations
    spins = 1 - 2 * bitstrings 
    evs = []
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
        evs.append(np.mean(obs_val))
    return np.array(evs)

# Execute the ensemble
E_x_qpu_total, E_y_qpu_total, E_z_qpu_total = 0, 0, 0

print(f"\nExecuting {num_rc_circuits} twirled circuits on Ankaa-3 ({shots_per_circ} shots each)...")
for i, prog in enumerate(pyquil_rc_progs):
    print(f"  -> Running circuit {i+1}/{num_rc_circuits}")
    
    # This correctly physically addresses the circuit using our mapping dictionary
    mapped_prog = address_qubits(prog, qubit_mapping)
    
    print("     [1/3] Measuring X basis...")
    res_x = run_mapped_basis(mapped_prog, 'X', qubit_mapping, qc, shots_per_circ)
    print("     [2/3] Measuring Y basis...")
    res_y = run_mapped_basis(mapped_prog, 'Y', qubit_mapping, qc, shots_per_circ)
    print("     [3/3] Measuring Z basis...")
    res_z = run_mapped_basis(mapped_prog, 'Z', qubit_mapping, qc, shots_per_circ)
    
    E_x_qpu_total += get_expectations(res_x, observables_x, shots_per_circ)
    E_y_qpu_total += get_expectations(res_y, observables_y, shots_per_circ)
    E_z_qpu_total += get_expectations(res_z, observables_z, shots_per_circ)

# Average out the expectation values
E_x_qpu = E_x_qpu_total / num_rc_circuits
E_y_qpu = E_y_qpu_total / num_rc_circuits
E_z_qpu = E_z_qpu_total / num_rc_circuits

E_final_qpu = np.concatenate([E_x_qpu, E_y_qpu, E_z_qpu])
print("\nAnkaa-3 QPU Evaluation Complete!\n")

# --- QPU Decoding & Local Search ---
cur_bits = [1 if val >= 0 else 0 for val in E_final_qpu]
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from Ankaa-3 before Local Search: {best_cut}")

for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post Ankaa-3 execution):", best_cut)

Preparing twirled ensemble for Rigetti Ankaa-3...

Executing 6 twirled circuits on Ankaa-3 (1666 shots each)...
  -> Running circuit 1/6
     [1/3] Measuring X basis...


/tmp/ipykernel_200518/4213199920.py:53: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  if hasattr(result, "readout_data"):
/tmp/ipykernel_200518/4213199920.py:54: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  return result.readout_data.get("ro") # PyQuil 4.x


     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 2/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 3/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 4/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 5/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 6/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...

Ankaa-3 QPU Evaluation Complete!

Initial Cut Size from Ankaa-3 before Local Search: 3763.4490249965584
Final Optimized Cut Size (Post Ankaa-3 execution): 6379.95530004793


In [18]:
# ==========================================
# 10. APPLY MAPPING & EXECUTE ON ANKAA-3
# ==========================================
from pyquil.quil import Program
#from pyquil.quilatom import Pragma
from pyquil.gates import H, RX, MEASURE
import numpy as np

print("Preparing twirled ensemble for Rigetti Ankaa-3...")

# 🏆 The Optimal Comprehensive Chain mapping (Highest T1/T2, f1Q, fRO, fISWAP)
qubit_mapping = {
    0: 2, 1: 3, 2: 4, 3: 11, 4: 18, 5: 19, 6: 12, 7: 5, 8: 6
}

qc = get_qc("Ankaa-3")
total_shots = 10000
# Distribute shots evenly across the RC ensemble
shots_per_circ = total_shots // num_rc_circuits

def run_mapped_basis(mapped_prog, basis, mapping, qc, shots):
    # FORCE the compiler to use our exact physical qubits, no auto-rerouting!
    prog = Program(f'PRAGMA INITIAL_REWIRE "NAIVE"')
    prog += mapped_prog
    
    # Declare the readout register (ro) sized to our logical circuit
    ro = prog.declare("ro", "BIT", len(mapping))
    
    # Append the basis transformation gates to the physical qubits
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
    # Measure the physical qubits into the logical indices of the readout register
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    exe = qc.compile(prog)
    result = qc.run(exe)
    
    # PyQuil 3.x+ syntax for extracting readout results
    if hasattr(result, "readout_data"):
        return result.readout_data.get("ro") # PyQuil 4.x
    else:
        return result.get_register_map().get("ro") # Older PyQuil 3.x

def get_expectations(bitstrings, qiskit_observables, shots):
    # Convert {0, 1} bitstrings to {+1, -1} spin expectations
    spins = 1 - 2 * bitstrings 
    evs = []
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
        evs.append(np.mean(obs_val))
    return np.array(evs)

# Execute the ensemble
E_x_qpu_total, E_y_qpu_total, E_z_qpu_total = 0, 0, 0

print(f"\nExecuting {num_rc_circuits} twirled circuits on Ankaa-3 ({shots_per_circ} shots each)...")
for i, prog in enumerate(pyquil_rc_progs):
    print(f"  -> Running circuit {i+1}/{num_rc_circuits}")
    
    # This correctly physically addresses the circuit using our mapping dictionary
    mapped_prog = address_qubits(prog, qubit_mapping)
    
    print("     [1/3] Measuring X basis...")
    res_x = run_mapped_basis(mapped_prog, 'X', qubit_mapping, qc, shots_per_circ)
    print("     [2/3] Measuring Y basis...")
    res_y = run_mapped_basis(mapped_prog, 'Y', qubit_mapping, qc, shots_per_circ)
    print("     [3/3] Measuring Z basis...")
    res_z = run_mapped_basis(mapped_prog, 'Z', qubit_mapping, qc, shots_per_circ)
    
    E_x_qpu_total += get_expectations(res_x, observables_x, shots_per_circ)
    E_y_qpu_total += get_expectations(res_y, observables_y, shots_per_circ)
    E_z_qpu_total += get_expectations(res_z, observables_z, shots_per_circ)

# Average out the expectation values
E_x_qpu = E_x_qpu_total / num_rc_circuits
E_y_qpu = E_y_qpu_total / num_rc_circuits
E_z_qpu = E_z_qpu_total / num_rc_circuits

E_final_qpu = np.concatenate([E_x_qpu, E_y_qpu, E_z_qpu])
print("\nAnkaa-3 QPU Evaluation Complete!\n")

# --- QPU Decoding & Local Search ---
cur_bits = [1 if val >= 0 else 0 for val in E_final_qpu]
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from Ankaa-3 before Local Search: {best_cut}")

for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post Ankaa-3 execution):", best_cut)

Preparing twirled ensemble for Rigetti Ankaa-3...

Executing 6 twirled circuits on Ankaa-3 (1666 shots each)...
  -> Running circuit 1/6
     [1/3] Measuring X basis...


/tmp/ipykernel_200518/1101383168.py:45: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  if hasattr(result, "readout_data"):
/tmp/ipykernel_200518/1101383168.py:46: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  return result.readout_data.get("ro") # PyQuil 4.x


     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 2/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 3/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 4/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 5/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 6/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...

Ankaa-3 QPU Evaluation Complete!

Initial Cut Size from Ankaa-3 before Local Search: 3649.6349320488566
Final Optimized Cut Size (Post Ankaa-3 execution): 6162.353818192175


In [19]:
# ==========================================
# 10. APPLY MAPPING & EXECUTE ON ANKAA-3
# ==========================================
from pyquil.quil import Program
#from pyquil.quilatom import Pragma
from pyquil.gates import H, RX, MEASURE
import numpy as np

print("Preparing twirled ensemble for Rigetti Ankaa-3...")

# 🏆 The Top 5 Optimal Comprehensive Chain Mappings
top_5_mappings = {
    # Highest overall composite scores (Great balance of high T1/T2 and >80% Fid)
    "Rank 1": {0: 2, 1: 3, 2: 4, 3: 11, 4: 18, 5: 19, 6: 12, 7: 5, 8: 6}, # Score: 427.74, Fid: 0.8211, T1: 27.74µs, T2: 18.78µs
    "Rank 2": {0: 2, 1: 3, 2: 4, 3: 5, 4: 12, 5: 19, 6: 18, 7: 11, 8: 10}, # Score: 413.91, Fid: 0.8139, T1: 27.08µs, T2: 18.78µs
    "Rank 3": {0: 2, 1: 3, 2: 4, 3: 5, 4: 12, 5: 11, 6: 18, 7: 19, 8: 20}, # Score: 413.33, Fid: 0.7935, T1: 27.74µs, T2: 18.78µs
    
    # These mappings dip into Qubit 9, dropping T2 slightly, but have incredible operational fidelity (~84-87%)
    "Rank 4": {0: 8, 1: 9, 2: 2, 3: 3, 4: 4, 5: 5, 6: 12, 7: 19, 8: 20}, # Score: 396.80, Fid: 0.8759, T1: 27.74µs, T2: 16.33µs
    "Rank 5": {0: 8, 1: 9, 2: 2, 3: 3, 4: 4, 5: 11, 6: 12, 7: 19, 8: 20}, # Score: 383.12, Fid: 0.8457, T1: 27.74µs, T2: 16.33µs
    
    # Alternative geometry shifting lower into the 20s
    "Rank 6": {0: 20, 1: 19, 2: 12, 3: 5, 4: 4, 5: 11, 6: 18, 7: 25, 8: 24}, # Score: 381.67, Fid: 0.8551, T1: 23.90µs, T2: 18.68µs
    
    # Back to the Qubit 9 / Qubit 18 cluster
    "Rank 7": {0: 8, 1: 9, 2: 2, 3: 3, 4: 4, 5: 5, 6: 12, 7: 19, 8: 18}, # Score: 381.27, Fid: 0.8416, T1: 27.74µs, T2: 16.33µs
    "Rank 8": {0: 10, 1: 17, 2: 18, 3: 11, 4: 4, 5: 5, 6: 12, 7: 19, 8: 20}, # Score: 377.73, Fid: 0.7428, T1: 27.08µs, T2: 18.78µs
    "Rank 9": {0: 9, 1: 2, 2: 3, 3: 4, 4: 5, 5: 12, 6: 19, 7: 18, 8: 17}, # Score: 377.61, Fid: 0.8335, T1: 27.74µs, T2: 16.33µs
    "Rank 10": {0: 8, 1: 9, 2: 2, 3: 3, 4: 4, 5: 11, 6: 18, 7: 19, 8: 12}, # Score: 377.37, Fid: 0.8330, T1: 27.74µs, T2: 16.33µs
    "Rank 11": {0: 18, 1: 17, 2: 10, 3: 11, 4: 4, 5: 5, 6: 12, 7: 19, 8: 20}, # Score: 377.33, Fid: 0.7420, T1: 27.08µs, T2: 18.78µs
    "Rank 12": {0: 5, 1: 12, 2: 19, 3: 18, 4: 11, 5: 4, 6: 3, 7: 2, 8: 9}, # Score: 376.95, Fid: 0.8321, T1: 27.74µs, T2: 16.33µs
    "Rank 13": {0: 8, 1: 9, 2: 2, 3: 3, 4: 4, 5: 11, 6: 18, 7: 19, 8: 20}, # Score: 376.54, Fid: 0.8312, T1: 27.74µs, T2: 16.33µs
    
    # Another variation dropping into the 20s
    "Rank 14": {0: 10, 1: 11, 2: 4, 3: 5, 4: 12, 5: 19, 6: 18, 7: 25, 8: 24}, # Score: 375.06, Fid: 0.8403, T1: 23.90µs, T2: 18.68µs
    
    # Final variant using Qubit 9
    "Rank 15": {0: 9, 1: 2, 2: 3, 3: 4, 4: 5, 5: 12, 6: 19, 7: 18, 8: 11}, # Score: 374.47, Fid: 0.8266, T1: 27.74µs, T2: 16.33µs
}

qc = get_qc("Ankaa-3")
total_shots = 10000
# Distribute shots evenly across the RC ensemble
shots_per_circ = total_shots // num_rc_circuits

def run_mapped_basis(mapped_prog, basis, mapping, qc, shots):
    # FORCE the compiler to use our exact physical qubits, no auto-rerouting!
    prog = Program('PRAGMA INITIAL_REWIRE "NAIVE"')
    prog += mapped_prog
    
    # Declare the readout register (ro) sized to our logical circuit
    ro = prog.declare("ro", "BIT", len(mapping))
    
    # Append the basis transformation gates to the physical qubits
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
    # Measure the physical qubits into the logical indices of the readout register
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    exe = qc.compile(prog)
    result = qc.run(exe)
    
    # PyQuil 3.x+ syntax for extracting readout results
    if hasattr(result, "readout_data"):
        return result.readout_data.get("ro") # PyQuil 4.x
    else:
        return result.get_register_map().get("ro") # Older PyQuil 3.x

def get_expectations(bitstrings, qiskit_observables, shots):
    # Convert {0, 1} bitstrings to {+1, -1} spin expectations
    spins = 1 - 2 * bitstrings 
    evs = []
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
        evs.append(np.mean(obs_val))
    return np.array(evs)

# Dictionary to store final results for the summary table
mapping_results = {}

print(f"\n=======================================================")
print(f" STARTING MULTI-MAPPING EXECUTION ON ANKAA-3")
print(f"=======================================================\n")

# Iterate through the top 5 mappings
for rank_name, qubit_mapping in top_5_mappings.items():
    print(f"\n--- Evaluating {rank_name} ---")
    print(f"Mapping: {qubit_mapping}")
    
    E_x_qpu_total, E_y_qpu_total, E_z_qpu_total = 0, 0, 0

    print(f"Executing {num_rc_circuits} twirled circuits ({shots_per_circ} shots each)...")
    for i, prog in enumerate(pyquil_rc_progs):
        # This correctly physically addresses the circuit using our mapping dictionary
        mapped_prog = address_qubits(prog, qubit_mapping)
        
        # Suppress repeated verbose output for circuits, keeping it clean
        res_x = run_mapped_basis(mapped_prog, 'X', qubit_mapping, qc, shots_per_circ)
        res_y = run_mapped_basis(mapped_prog, 'Y', qubit_mapping, qc, shots_per_circ)
        res_z = run_mapped_basis(mapped_prog, 'Z', qubit_mapping, qc, shots_per_circ)
        
        E_x_qpu_total += get_expectations(res_x, observables_x, shots_per_circ)
        E_y_qpu_total += get_expectations(res_y, observables_y, shots_per_circ)
        E_z_qpu_total += get_expectations(res_z, observables_z, shots_per_circ)

    # Average out the expectation values
    E_x_qpu = E_x_qpu_total / num_rc_circuits
    E_y_qpu = E_y_qpu_total / num_rc_circuits
    E_z_qpu = E_z_qpu_total / num_rc_circuits

    E_final_qpu = np.concatenate([E_x_qpu, E_y_qpu, E_z_qpu])
    print(f"QPU Execution Complete for {rank_name}.")

    # --- QPU Decoding & Local Search ---
    cur_bits = [1 if val >= 0 else 0 for val in E_final_qpu]
    cur_partition = [set(), set()]
    for i, bit in enumerate(cur_bits):
        cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
            
    best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    initial_cut = best_cut
    best_bits = cur_bits.copy()

    print(f"  -> Initial Cut Size (Before Local Search): {initial_cut}")

    for node in range(num_nodes):
        swapped_bits = best_bits.copy()
        swapped_bits[node] = 1 - swapped_bits[node] 
        
        cur_partition = [set(), set()]
        for i, bit in enumerate(swapped_bits):
            cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
                
        cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
        if cut_size > best_cut:
            best_cut = cut_size
            best_bits = swapped_bits

    final_cut = best_cut
    print(f"  -> Final Optimized Cut Size: {final_cut}")
    
    # Save to results dictionary
    mapping_results[rank_name] = {
        "Initial Cut": initial_cut,
        "Final Cut": final_cut
    }

# ==========================================
# 11. FINAL SUMMARIZING TABLE
# ==========================================
print("\n" + "="*60)
print(f"{'ANKAA-3 MULTI-MAPPING EXECUTION SUMMARY':^60}")
print("="*60)
print(f"{'Mapping Rank':<15} | {'Initial Cut Size':<20} | {'Optimized Cut Size':<20}")
print("-" * 60)

for rank_name, results in mapping_results.items():
    init_cut = results["Initial Cut"]
    fin_cut = results["Final Cut"]
    print(f"{rank_name:<15} | {init_cut:<20} | {fin_cut:<20}")

print("="*60 + "\n")

Preparing twirled ensemble for Rigetti Ankaa-3...

 STARTING MULTI-MAPPING EXECUTION ON ANKAA-3


--- Evaluating Rank 1 ---
Mapping: {0: 2, 1: 3, 2: 4, 3: 11, 4: 18, 5: 19, 6: 12, 7: 5, 8: 6}
Executing 6 twirled circuits (1666 shots each)...


/tmp/ipykernel_200518/2916012478.py:70: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  if hasattr(result, "readout_data"):
/tmp/ipykernel_200518/2916012478.py:71: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  return result.readout_data.get("ro") # PyQuil 4.x


QPU Execution Complete for Rank 1.
  -> Initial Cut Size (Before Local Search): 4183.0165450996565
  -> Final Optimized Cut Size: 6024.04401943519

--- Evaluating Rank 2 ---
Mapping: {0: 2, 1: 3, 2: 4, 3: 5, 4: 12, 5: 19, 6: 18, 7: 11, 8: 10}
Executing 6 twirled circuits (1666 shots each)...
QPU Execution Complete for Rank 2.
  -> Initial Cut Size (Before Local Search): 2834.9821642318143
  -> Final Optimized Cut Size: 6156.947982146791

--- Evaluating Rank 3 ---
Mapping: {0: 2, 1: 3, 2: 4, 3: 5, 4: 12, 5: 11, 6: 18, 7: 19, 8: 20}
Executing 6 twirled circuits (1666 shots each)...
QPU Execution Complete for Rank 3.
  -> Initial Cut Size (Before Local Search): 3455.8135426921976
  -> Final Optimized Cut Size: 6105.120028412934

--- Evaluating Rank 4 ---
Mapping: {0: 8, 1: 9, 2: 2, 3: 3, 4: 4, 5: 5, 6: 12, 7: 19, 8: 20}
Executing 6 twirled circuits (1666 shots each)...
QPU Execution Complete for Rank 4.
  -> Initial Cut Size (Before Local Search): 3465.7932307162287
  -> Final Optimized C

KeyboardInterrupt: 